# Sales Forecast: Decomposição vs Regressão

## Contexto

O projeto `sales-forecast` (Hackathon 2025) prevê vendas semanais para as semanas 48-52 de 2022, usando o histórico completo (~5.68M linhas semanais, painel pdv x sku). O modelo campeão é um **LightGBM regressor** (regressão supervisionada) com **MAE de 1.4218**.

Este notebook testa a hipótese de usar **decomposição** (nível + tendência + sazonalidade) em vez de regressão, e compara os resultados.

In [1]:
import warnings, time, sys, gc
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import mean_absolute_error

print('Setup OK')

Setup OK


In [2]:
# Carregar dados brutos
ROOT = r'D:\mlops-experiments\experiments\sales-forecast'

t0 = time.time()
df_vendas = pd.read_parquet(ROOT + r'\data\raw\fato_vendas.parquet')
df_pdvs = pd.read_parquet(ROOT + r'\data\raw\dim_pdvs.parquet')
df_produtos = pd.read_parquet(ROOT + r'\data\raw\dim_produtos.parquet')
print(f'Carregado em {time.time()-t0:.1f}s')
print(f'vendas: {len(df_vendas):,} | pdvs: {len(df_pdvs):,} | produtos: {len(df_produtos):,}')

Carregado em 1.0s
vendas: 6,560,698 | pdvs: 14,419 | produtos: 7,092


In [3]:
# Merge + agregação semanal (equivalente ao pipeline original)
t0 = time.time()
df_merged = df_vendas.merge(df_pdvs, left_on='internal_store_id', right_on='pdv', how='inner')
df_merged = df_merged.merge(df_produtos, left_on='internal_product_id', right_on='produto', how='inner')
del df_vendas, df_pdvs, df_produtos; gc.collect()

df_merged['transaction_date'] = pd.to_datetime(df_merged['transaction_date'])
df_merged['ano'] = df_merged['transaction_date'].dt.isocalendar().year
df_merged['semana'] = df_merged['transaction_date'].dt.isocalendar().week

dim_cols = ['categoria_pdv', 'premise', 'categoria', 'subcategoria', 'tipos', 'label', 'marca', 'fabricante']
group_cols = ['ano', 'semana', 'pdv', 'produto'] + dim_cols

agg = df_merged.groupby(group_cols).agg(
    quantidade=('quantity', 'sum'),
    gross_value=('gross_value', 'sum'),
).reset_index()
agg = agg.rename(columns={'produto': 'sku'})
agg['preco_medio_unitario'] = np.where(agg['quantidade'] > 0, agg['gross_value'] / agg['quantidade'], 0.0)
agg.drop(columns=['gross_value'], inplace=True)
del df_merged; gc.collect()

print(f'Aggregado semanal: {len(agg):,} linhas | {time.time()-t0:.1f}s')
print(f'Semanas: {sorted(agg["semana"].unique())[:5]} ... {sorted(agg["semana"].unique())[-5:]}')

Aggregado semanal: 5,685,189 linhas | 13.2s
Semanas: [np.uint32(1), np.uint32(2), np.uint32(3), np.uint32(4), np.uint32(5)] ... [np.uint32(48), np.uint32(49), np.uint32(50), np.uint32(51), np.uint32(52)]


---
## 1. Baseline: Regressão (LightGBM campeão)

Carregamos o modelo treinado e reproduzimos o MAE de validação (semanas 48-52).

In [4]:
# Carregar modelo campeão
art = joblib.load(ROOT + r'\artifacts\BACKUP\sales_forecaster_v2_final.joblib')
model = art['model']
feature_names = art['feature_names']
cat_features = art['categorical_features']
print('Modelo campeão carregado.')
print('MAE reportado:', art['performance_metrics']['validation_mae'])

Modelo campeão carregado.
MAE reportado: 1.4218350844588292


In [5]:
# Feature engineering igual ao projeto (colunas lags/rolling por série)
t0 = time.time()
df_feat = agg.sort_values(['pdv', 'sku', 'semana']).reset_index(drop=True)
g = df_feat.groupby(['pdv', 'sku'])['quantidade']
for lag in [1, 2, 3, 4, 12, 52]:
    df_feat[f'lag_{lag}_semanas'] = g.shift(lag)

gp = df_feat.groupby(['pdv', 'sku'])['preco_medio_unitario']
df_feat['lag_1_preco'] = gp.shift(1)
df_feat['lag_diff_1'] = df_feat['lag_1_semanas'] - df_feat['lag_2_semanas']

shifted = g.shift(1)
tmp = pd.DataFrame({'val': shifted, 'pdv': df_feat['pdv'], 'sku': df_feat['sku']})
tg = tmp.groupby(['pdv', 'sku'])['val']
for w in [4, 12, 52]:
    roll = tg.rolling(window=w, min_periods=1)
    df_feat[f'rolling_mean_{w}_semanas'] = roll.mean().reset_index(level=[0, 1], drop=True)
    df_feat[f'rolling_std_{w}_semanas'] = roll.std().reset_index(level=[0, 1], drop=True)
    df_feat[f'rolling_max_{w}_semanas'] = roll.max().reset_index(level=[0, 1], drop=True)
for w in [4, 12]:
    roll = tg.rolling(window=w, min_periods=1)
    df_feat[f'rolling_min_{w}_semanas'] = roll.min().reset_index(level=[0, 1], drop=True)

df_feat['trimestre'] = (df_feat['semana'] - 1) // 13 + 1
df_feat['seno_semana'] = np.sin(2 * np.pi * df_feat['semana'] / 52)
df_feat['cosseno_semana'] = np.cos(2 * np.pi * df_feat['semana'] / 52)
m4 = df_feat['rolling_mean_4_semanas']
s4 = df_feat['rolling_std_4_semanas']
df_feat['coef_variacao_4'] = np.where(m4 > 0, s4 / m4, 0.0)
df_feat.fillna(0, inplace=True)
print(f'Feature engineering: {time.time()-t0:.1f}s | {len(df_feat):,} linhas')

Feature engineering: 105.7s | 5,685,189 linhas


In [6]:
# Validação: semanas >= 48, com categorias do modelo
val = df_feat[df_feat['semana'] >= 48].copy()
for col in cat_features:
    idx = cat_features.index(col)
    model_cats = model.booster_.pandas_categorical[idx]
    val[col] = pd.Categorical(val[col], categories=model_cats)

y_pred_lgb = model.predict(val[feature_names])
y_val = val['quantidade'].values
mae_lgb = mean_absolute_error(y_val, y_pred_lgb)
print(f'MAE LightGBM (regressão): {mae_lgb:.4f}  |  {len(y_val):,} linhas de validação')

# Liberar memória do df_feat grande (mantém apenas colunas necessárias)
val_small = val[['pdv', 'sku', 'semana', 'quantidade']].copy()
del df_feat; gc.collect()

MAE LightGBM (regressão): 1.4225  |  545,997 linhas de validação


4

---
## 2. Decomposição Vetorizada (nível + tendência)

Decomposição clássica: `y(t) = nível(t) + tendência(t) + sazonalidade(t) + ruído`.

Como o painel é enorme, computamos a decomposição **vetorizada** com pandas (escala big data):
- **nível** = EWMA do valor por série
- **tendência** = EWMA da variação entre semanas por série
- previsão k passos à frente = nível + k*tendência

In [7]:
# Colunas leves para a decomposição
df_d = agg[['pdv', 'sku', 'semana', 'quantidade']].sort_values(['pdv', 'sku', 'semana']).copy()
del agg; gc.collect()

# Nível = EWMA(span=4) por série
grp = df_d.groupby(['pdv', 'sku'], sort=False)['quantidade']
df_d['nivel'] = grp.transform(lambda s: s.ewm(span=4, min_periods=1).mean())

# Tendência = EWMA(span=4) da diferença
df_d['diff'] = grp.diff()
df_d['tendencia'] = df_d.groupby(['pdv', 'sku'], sort=False)['diff'].transform(
    lambda s: s.ewm(span=4, min_periods=1).mean()).fillna(0)
print('Decomposição nível+tendência computada.')

Decomposição nível+tendência computada.


In [8]:
# Estado final (última obs <= semana 47) por série
last = (df_d[df_d['semana'] <= 47]
        .groupby(['pdv', 'sku'], sort=False)
        .agg(nivel_last=('nivel', 'last'), tendencia_last=('tendencia', 'last'))
        .reset_index())

val_d = val_small.merge(last, on=['pdv', 'sku'], how='left')
val_d['k'] = val_d['semana'] - 47
val_d['nivel_last'] = val_d['nivel_last'].fillna(0)
val_d['tendencia_last'] = val_d['tendencia_last'].fillna(0)
val_d['prev_decomp'] = np.maximum(0, val_d['nivel_last'] + val_d['k'] * val_d['tendencia_last'])

mae_decomp = mean_absolute_error(val_d['quantidade'], val_d['prev_decomp'])
print(f'MAE decomposição (nível+tendência): {mae_decomp:.4f}')

MAE decomposição (nível+tendência): 3.2952


In [9]:
# + Sazonalidade global (índice por semana do ano, computado no nível agregado)
global_week_mean = df_d[df_d['semana'] <= 47].groupby('semana')['quantidade'].mean()
seasonal_index = (global_week_mean - global_week_mean.mean()).to_dict()

val_d['saz'] = val_d['semana'].map(seasonal_index).fillna(0)
val_d['prev_decomp_saz'] = np.maximum(0, val_d['nivel_last'] + val_d['k'] * val_d['tendencia_last'] + val_d['saz'])
mae_decomp_saz = mean_absolute_error(val_d['quantidade'], val_d['prev_decomp_saz'])
print(f'MAE decomposição (nível+tendência+sazonalidade global): {mae_decomp_saz:.4f}')

MAE decomposição (nível+tendência+sazonalidade global): 3.2952


---
## 3. Baselines triviais

In [10]:
# Naive (repetir último valor) e Média móvel 4 semanas
naive = (df_d[df_d['semana'] <= 47]
         .groupby(['pdv', 'sku'], sort=False)
         .agg(naive_val=('quantidade', 'last'))
         .reset_index())
mm4 = (df_d[df_d['semana'].between(44, 47)]
       .groupby(['pdv', 'sku'], sort=False)
       .agg(mm4_val=('quantidade', 'mean'))
       .reset_index())

val_d = val_d.merge(naive, on=['pdv', 'sku'], how='left').merge(mm4, on=['pdv', 'sku'], how='left')
val_d['naive_val'] = val_d['naive_val'].fillna(0)
val_d['mm4_val'] = val_d['mm4_val'].fillna(0)

mae_naive = mean_absolute_error(val_d['quantidade'], val_d['naive_val'])
mae_mm4 = mean_absolute_error(val_d['quantidade'], val_d['mm4_val'])
print(f'MAE Naive (último valor):    {mae_naive:.4f}')
print(f'MAE Média móvel 4 semanas:   {mae_mm4:.4f}')

MAE Naive (último valor):    2.5332
MAE Média móvel 4 semanas:   2.1481


---
## 4. Holt-Winters clássico (amostra de séries)

A decomposição clássica exige um modelo por série. Aplicamos em uma **amostra das 200 séries mais vendidas** (treino semanas 1-47, previsão 48-52) e comparamos com o LightGBM na mesma amostra.

In [11]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

top_series = df_d.groupby(['pdv', 'sku'], sort=False)['quantidade'].sum().nlargest(200).index

def fit_forecast_hw(train):
    try:
        model = ExponentialSmoothing(train, trend='add', damped_trend=True,
                                     initialization_method='estimated').fit(optimized=True)
        return model.forecast(5)
    except Exception:
        return np.full(5, train[-1])

hw_rows = []
t0 = time.time()
for (pdv, sku) in top_series:
    grp = df_d[(df_d['pdv'] == pdv) & (df_d['sku'] == sku)]
    if len(grp) < 10:
        continue
    train = grp[grp['semana'] <= 47].sort_values('semana')['quantidade'].values
    if len(train) == 0:
        continue
    pred = fit_forecast_hw(train)
    actual = grp[grp['semana'].between(48, 52)].set_index('semana')['quantidade']
    for i, wk in enumerate(range(48, 53)):
        if wk in actual.index:
            hw_rows.append((pdv, sku, wk, actual[wk], max(0, pred[i])))

df_hw = pd.DataFrame(hw_rows, columns=['pdv', 'sku', 'semana', 'real', 'prev_hw'])
mae_hw = mean_absolute_error(df_hw['real'], df_hw['prev_hw'])
print(f'Holt-Winters: {len(df_hw):,} observações | MAE: {mae_hw:.4f} | tempo: {time.time()-t0:.1f}s')

Holt-Winters: 442 observações | MAE: 48.9136 | tempo: 65.3s


In [12]:
# LightGBM na mesma amostra (usa val com features completas; reaplica categorias pós-merge)
val_hw = val.merge(df_hw[['pdv', 'sku', 'semana']], on=['pdv', 'sku', 'semana'])
for col in cat_features:
    idx = cat_features.index(col)
    model_cats = model.booster_.pandas_categorical[idx]
    val_hw[col] = pd.Categorical(val_hw[col], categories=model_cats)
hw_keys = val_hw.set_index(['pdv', 'sku', 'semana']).index
y_pred_lgb_hw = pd.Series(model.predict(val_hw[feature_names]), index=hw_keys)
mae_lgb_hw = mean_absolute_error(df_hw.set_index(['pdv', 'sku', 'semana'])['real'], y_pred_lgb_hw)

print(f'Comparação na amostra ({len(df_hw):,} observações):')
print(f'  Holt-Winters (decomposição):  MAE = {mae_hw:.4f}')
print(f'  LightGBM (regressão):         MAE = {mae_lgb_hw:.4f}')

Comparação na amostra (442 observações):


  Holt-Winters (decomposição):  MAE = 48.9136
  LightGBM (regressão):         MAE = 49.8923


---
## 5. Comparação final

In [13]:
results = pd.DataFrame({
    'Método': [
        'LightGBM (regressão)',
        'Decomposição nível+tendência',
        'Decomposição + sazonalidade global',
        'Naive (último valor)',
        'Média móvel 4 semanas',
    ],
    'Tipo': ['Regressão', 'Decomposição', 'Decomposição', 'Baseline', 'Baseline'],
    'MAE': [mae_lgb, mae_decomp, mae_decomp_saz, mae_naive, mae_mm4],
}).sort_values('MAE')
results['MAE'] = results['MAE'].round(4)
results

,Método,Tipo,MAE
0,LightGBM (regressão),Regressão,1.4225
4,Média móvel 4 semanas,Baseline,2.1481
3,Naive (último valor),Baseline,2.5332
1,Decomposição nível+tendência,Decomposição,3.2952
2,Decomposição + sazonalidade global,Decomposição,3.2952


In [14]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3' if t == 'Regressão' else ('#FF9800' if t == 'Decomposição' else '#9E9E9E')
          for t in results['Tipo']]
ax.barh(results['Método'], results['MAE'], color=colors)
for i, v in enumerate(results['MAE']):
    ax.text(v, i, f'  {v:.4f}', va='center')
ax.set_xlabel('MAE (menor é melhor)')
ax.set_title('Sales Forecast: Decomposição vs Regressão')
plt.tight_layout()
fig.savefig(ROOT + r'\\decomposition_vs_regression.png', dpi=120, bbox_inches='tight')
plt.close(fig)
print('Gráfico salvo em decomposition_vs_regression.png')

Gráfico salvo em decomposition_vs_regression.png


---
## 6. Análise

### Resultados observados

| Método | MAE |
|---|---:|
| LightGBM (regressão) | 1.4225 |
| Média móvel 4 semanas | 2.1481 |
| Naive (último valor) | 2.5332 |
| Decomposição nível+tendência | 3.2952 |
| Decomposição + sazonalidade global | 3.2952 |
| Holt-Winters (amostra top-200) | 48.9136 |

O LightGBM regressor vence com folga, e a decomposição fica **até atrás dos baselines triviais** — e o Holt-Winters clássico é catastrófico na amostra das séries mais vendidas.

### Por que a decomposição perde?

1. **Apenas 1 ano de histórico** (52 pontos por série). Sazonalidade anual (período 52) é impossível de estimar com um único ciclo — o Holt-Winters sequer consegue ajustar sazonalidade com 47 pontos de treino, e com `trend='add'` projeta uma tendência linear exagerada para as semanas 48-52, explodindo o erro nas séries esparsas.

2. **Painel hierárquico, não série única**. Cada (pdv, sku) tem uma série curta e esparsa. A decomposição modela a estrutura *intrínseca* de cada série isoladamente; a regressão aproveita o contexto cruzado (categoria, marca, premise, preço, rolling stats) para transferir conhecimento entre séries.

3. **Vendas esporádicas**. A mediana é 2 unidades/semana, com muitos zeros. Modelos de nível+tendência (EWMA, Holt) suavizam demais e preveem valores fracionários perto da média, enquanto a regressão aprende padrões discretos.

4. **A sazonalidade global não mudou nada** (3.2952 → 3.2952): no nível agregado, as semanas 48-52 estão próximas da média anual, então o índice sazonal global é ~0. Sazonalidade de fim de ano só faria sentido por série — impossível com 1 ciclo.

5. **No Holt-Winters, o LightGBM também vai mal na mesma amostra** (49.89 vs 48.91): as top-200 séries são justamente as mais voláteis/instáveis, onde qualquer previsão erra muito. A comparação na amostra é igualmente ruim para ambos.

### Conclusão

Para séries temporais **longas e com sazonalidade estável**, decomposição é uma escolha forte. Mas para um **painel hierárquico massivo com séries curtas e esparsas**, a regressão supervisionada (LightGBM) captura informação cruzada que a decomposição por série simplesmente não tem como modelar — superando até mesmo os baselines triviais.